# 01 — Wheels et publication sur PyPI

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :
- expliquer la différence entre sdist et wheel ;
- configurer un projet avec `pyproject.toml` (PEP 517/518/621) ;
- construire un package avec `build` ;
- publier sur TestPyPI puis PyPI avec `twine` ;
- gérer les versions avec les conventions SemVer ;
- utiliser les extras, entry points et classifiers.

## Prérequis — ce que vous connaissez déjà

Ce notebook s'adresse à un développeur Python **confirmé**. Vous maîtrisez déjà :
- les modules et packages Python (`__init__.py`, imports relatifs) ;
- les environnements virtuels (`venv`, `pip`) ;
- la ligne de commande et le terminal ;
- la configuration de base d'un projet Python.

## Plan

1. Pourquoi empaqueter ?
2. sdist vs wheel
3. `pyproject.toml` — le standard moderne
4. Structure d'un projet type
5. Construire avec `python -m build`
6. Publier avec `twine`
7. Entry points (CLI)
8. Extras et dépendances optionnelles
9. Versionning et SemVer
10. Synthèse
11. Exercices
12. Ressources

---

## 1. Pourquoi empaqueter ?

| Objectif | Solution |
|---|---|
| Partager du code avec des collègues | Package interne |
| Distribuer un outil open-source | PyPI |
| Installer un CLI en une commande | `pip install mon-outil` |
| Gérer les dépendances proprement | `pyproject.toml` |
| Assurer la reproductibilité | Version pinning + wheel |

---

## 2. sdist vs wheel

| Critère | sdist (Source Distribution) | wheel (Built Distribution) |
|---|---|---|
| Extension | `.tar.gz` | `.whl` |
| Contenu | Code source + setup | Code pré-compilé |
| Installation | Nécessite compilation | Extraction directe |
| Vitesse d'installation | Lente | **Rapide** |
| Extensions C | Compilées à l'install | Pré-compilées |
| PEP | 517 | 427 |

In [ ]:
# Un fichier .whl est un zip renommé
print("Anatomie d'un nom de wheel :")
print("  mon_package-1.2.3-py3-none-any.whl")
print("  │            │     │   │    │")
print("  │            │     │   │    └─ plateforme (any = universel)")
print("  │            │     │   └────── ABI (none = Python pur)")
print("  │            │     └────────── implémentation (py3 = Python 3)")
print("  │            └──────────────── version")
print("  └───────────────────────────── nom du package")

---

## 3. `pyproject.toml` — le standard moderne

Depuis les PEP 517, 518 et 621, `pyproject.toml` est le fichier de configuration **unique** pour un projet Python. Il remplace `setup.py`, `setup.cfg` et `MANIFEST.in`.

In [ ]:
pyproject_exemple = '''
[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "mon-super-package"
version = "1.0.0"
description = "Un package Python exemplaire"
readme = "README.md"
license = {text = "MIT"}
requires-python = ">=3.10"
authors = [
    {name = "Alice Dupont", email = "alice@example.com"},
]
classifiers = [
    "Programming Language :: Python :: 3",
    "License :: OSI Approved :: MIT License",
    "Operating System :: OS Independent",
]
dependencies = [
    "requests>=2.28",
    "click>=8.0",
]

[project.optional-dependencies]
dev = ["pytest", "ruff", "mypy"]
docs = ["sphinx", "furo"]

[project.scripts]
mon-cli = "mon_package.cli:main"

[project.urls]
Homepage = "https://github.com/alice/mon-super-package"
Documentation = "https://mon-super-package.readthedocs.io"
'''

print(pyproject_exemple)

### Les build backends

| Backend | Mainteneur | Points forts |
|---|---|---|
| `hatchling` | Ofek Lev | Moderne, rapide, extensible |
| `setuptools` | PyPA | Historique, très large support |
| `flit-core` | Thomas Kluyver | Minimal, idéal pour Python pur |
| `maturin` | PyO3 | Pour les extensions Rust |
| `pdm-backend` | Frost Ming | Intégré à PDM |

### Champs obligatoires vs recommandés

| Champ | Obligatoire | Description |
|---|---|---|
| `name` | Oui | Nom sur PyPI (unique) |
| `version` | Oui | Version SemVer |
| `description` | Recommandé | Description courte (1 ligne) |
| `readme` | Recommandé | Fichier README |
| `license` | Recommandé | Licence |
| `requires-python` | Recommandé | Version Python minimale |
| `dependencies` | Non | Dépendances runtime |
| `authors` | Recommandé | Auteurs |

---

## 4. Structure d'un projet type

In [ ]:
structure = '''
mon-super-package/
├── pyproject.toml
├── README.md
├── LICENSE
├── src/
│   └── mon_package/
│       ├── __init__.py
│       ├── core.py
│       └── cli.py
├── tests/
│   ├── __init__.py
│   ├── test_core.py
│   └── test_cli.py
└── docs/
    └── index.md
'''
print(structure)

### Layout `src/` vs layout plat

| Layout | Avantage | Inconvénient |
|---|---|---|
| `src/mon_package/` | Impossible d'importer accidentellement le code non installé | Un niveau de dossier en plus |
| `mon_package/` (plat) | Plus simple | Risque de conflits d'import |

**Recommandation :** utilisez le layout `src/` pour les packages publiés.

---

## 5. Construire avec `python -m build`

> **Installation :** `pip install build`

```bash
# Construire sdist + wheel
python -m build

# Résultat dans dist/
# dist/mon_super_package-1.0.0.tar.gz   (sdist)
# dist/mon_super_package-1.0.0-py3-none-any.whl   (wheel)
```

In [ ]:
# Inspecter le contenu d'un wheel (c'est un zip)
print("""
$ unzip -l dist/mon_super_package-1.0.0-py3-none-any.whl
  mon_package/__init__.py
  mon_package/core.py
  mon_package/cli.py
  mon_super_package-1.0.0.dist-info/METADATA
  mon_super_package-1.0.0.dist-info/WHEEL
  mon_super_package-1.0.0.dist-info/RECORD
  mon_super_package-1.0.0.dist-info/entry_points.txt
""")

### Vérifier le package avant publication

```bash
# Vérifier la structure du wheel
pip install check-wheel-contents
check-wheel-contents dist/*.whl

# Vérifier les métadonnées
twine check dist/*
```

---

## 6. Publier avec `twine`

> **Installation :** `pip install twine`

### Étape 1 : tester sur TestPyPI

```bash
# Créer un compte sur https://test.pypi.org/
# Puis publier
twine upload --repository testpypi dist/*

# Tester l'installation
pip install --index-url https://test.pypi.org/simple/ mon-super-package
```

### Étape 2 : publier sur PyPI

```bash
# Créer un compte sur https://pypi.org/
# Utiliser un token API (pas votre mot de passe !)
twine upload dist/*
```

### Authentification avec des tokens

Depuis 2024, PyPI **exige** l'authentification par **token API** ou **Trusted Publisher** (GitHub Actions).

```
# ~/.pypirc
[pypi]
username = __token__
password = pypi-AgEI...

[testpypi]
username = __token__
password = pypi-AgEI...
```

### Trusted Publishers (GitHub Actions)

```yaml
# .github/workflows/publish.yml
name: Publish to PyPI
on:
  release:
    types: [published]

permissions:
  id-token: write

jobs:
  publish:
    runs-on: ubuntu-latest
    environment: release
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
      - run: pip install build
      - run: python -m build
      - uses: pypa/gh-action-pypi-publish@release/v1
```

Avec Trusted Publishers, aucun token n'est stocké dans les secrets GitHub.

---

## 7. Entry points (CLI)

Les entry points permettent de créer des **commandes CLI** installées automatiquement par pip.

In [ ]:
# Dans pyproject.toml :
config_entrypoint = '''
[project.scripts]
mon-cli = "mon_package.cli:main"
'''
print(config_entrypoint)

# Après pip install, la commande 'mon-cli' est disponible
# Elle appelle mon_package.cli.main()

In [ ]:
# Exemple de module cli.py
cli_example = '''
# src/mon_package/cli.py
import click

@click.command()
@click.option("--name", default="World", help="Nom à saluer")
def main(name: str) -> None:
    """Salue quelqu'un."""
    print(f"Hello, {name}!")

if __name__ == "__main__":
    main()
'''
print(cli_example)

### Plugins via entry points

```toml
[project.entry-points."mon_app.plugins"]
csv = "mon_package.plugins.csv:CsvPlugin"
json = "mon_package.plugins.json:JsonPlugin"
```

```python
# Découverte des plugins
from importlib.metadata import entry_points

plugins = entry_points(group="mon_app.plugins")
for ep in plugins:
    plugin_class = ep.load()
    print(f"Plugin {ep.name}: {plugin_class}")
```

---

## 8. Extras et dépendances optionnelles

In [ ]:
extras_example = '''
[project.optional-dependencies]
dev = [
    "pytest>=7.0",
    "ruff>=0.4",
    "mypy>=1.10",
]
docs = [
    "sphinx>=7.0",
    "furo",
]
all = ["mon-package[dev,docs]"]
'''
print(extras_example)

# Installation :
# pip install mon-package[dev]      → avec les outils de dev
# pip install mon-package[dev,docs] → dev + docs
# pip install mon-package[all]      → tout

---

## 9. Versionning et SemVer

**Semantic Versioning** (SemVer) : `MAJOR.MINOR.PATCH`

| Changement | Incrément | Exemple |
|---|---|---|
| Bug fix, correctif | PATCH | 1.2.3 → 1.2.4 |
| Nouvelle fonctionnalité (rétrocompatible) | MINOR | 1.2.3 → 1.3.0 |
| Changement cassant (API) | MAJOR | 1.2.3 → 2.0.0 |
| Pré-release | Suffixe | 1.3.0a1, 1.3.0b2, 1.3.0rc1 |

### Version dynamique avec hatch

In [ ]:
# pyproject.toml avec version dynamique
dynamic_version = '''
[project]
name = "mon-package"
dynamic = ["version"]

[tool.hatch.version]
path = "src/mon_package/__init__.py"
'''
print(dynamic_version)

# src/mon_package/__init__.py
print('__version__ = "1.2.3"')

### `importlib.metadata` — lire la version à runtime

In [ ]:
from importlib.metadata import version, PackageNotFoundError

for pkg in ["pip", "setuptools"]:
    try:
        v = version(pkg)
        print(f"{pkg} : {v}")
    except PackageNotFoundError:
        print(f"{pkg} : non installé")

---

## 10. Synthèse

| Outil | Rôle |
|---|---|
| `pyproject.toml` | Configuration unique du projet |
| `python -m build` | Construire sdist + wheel |
| `twine upload` | Publier sur PyPI |
| `twine check` | Vérifier les métadonnées |
| `[project.scripts]` | Entry points CLI |
| `[project.optional-dependencies]` | Extras |

**Règles à retenir :**
- `pyproject.toml` remplace `setup.py` et `setup.cfg`.
- Publiez toujours un wheel (`-py3-none-any.whl`) en plus du sdist.
- Testez sur TestPyPI avant de publier sur PyPI.
- Utilisez des tokens API, jamais votre mot de passe.
- Suivez SemVer pour le versionning.
- Layout `src/` pour les packages publiés.

---

## 11. Exercices

### Exercice 1 — Écrire un `pyproject.toml` *(facile)*

Écrire un `pyproject.toml` complet pour un package fictif `calculette` qui :
- utilise `hatchling` comme backend ;
- dépend de `click>=8.0` ;
- a un entry point CLI `calc` ;
- supporte Python 3.10+.

In [ ]:
# Votre code ici (affichez le contenu du pyproject.toml)


<details>
<summary>📖 Voir la correction</summary>

```python
print('''
[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "calculette"
version = "0.1.0"
description = "Une calculette en ligne de commande"
requires-python = ">=3.10"
license = {text = "MIT"}
dependencies = ["click>=8.0"]

[project.scripts]
calc = "calculette.cli:main"

[project.optional-dependencies]
dev = ["pytest", "ruff"]
''')
```

</details>

### Exercice 2 — Inspecter un wheel *(moyen)*

Écrire un script qui :
1. Prend le chemin d'un fichier `.whl` en argument ;
2. Lit le `METADATA` dans le zip ;
3. Affiche le nom, la version, les dépendances, et le Python requis.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Wheels_et_publication", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import zipfile
import email.parser
import glob
import os

def inspecter_wheel(chemin_whl: str) -> dict:
    with zipfile.ZipFile(chemin_whl) as zf:
        metadata_path = [n for n in zf.namelist() if n.endswith("/METADATA")][0]
        metadata_text = zf.read(metadata_path).decode("utf-8")

    parser = email.parser.Parser()
    meta = parser.parsestr(metadata_text)

    return {
        "name": meta.get("Name"),
        "version": meta.get("Version"),
        "requires_python": meta.get("Requires-Python"),
        "dependencies": meta.get_all("Requires-Dist") or [],
    }

# Chercher un wheel dans l'environnement
import site
site_packages = site.getsitepackages()[0] if site.getsitepackages() else ""
print(f"Exemple avec un package installé :")
print(f"(en production, passer un chemin .whl)")
```

</details>

### Exercice 3 — Script de release *(moyen)*

Écrire un script Python qui automatise le processus de release :
1. Lit la version depuis `__init__.py` ;
2. Vérifie que le CHANGELOG mentionne cette version ;
3. Construit avec `python -m build` ;
4. Vérifie avec `twine check` ;
5. Affiche la commande `twine upload` (sans l'exécuter).

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Wheels_et_publication", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
import re
import subprocess
import sys

def lire_version(chemin_init: str) -> str:
    with open(chemin_init) as f:
        match = re.search(r'__version__\s*=\s*"([^"]+)"', f.read())
    if not match:
        raise ValueError(f"Version non trouvée dans {chemin_init}")
    return match.group(1)

def verifier_changelog(chemin: str, version: str) -> bool:
    with open(chemin) as f:
        return version in f.read()

def release(init_path: str, changelog_path: str):
    version = lire_version(init_path)
    print(f"Version : {version}")

    if not verifier_changelog(changelog_path, version):
        print(f"ERREUR : {version} non trouvée dans {changelog_path}")
        sys.exit(1)

    print("Construction...")
    subprocess.run([sys.executable, "-m", "build"], check=True)

    print("Vérification...")
    subprocess.run(["twine", "check", "dist/*"], check=True)

    print(f"Prêt ! Exécutez :")
    print(f"  twine upload dist/mon_package-{version}*")

# Exemple (ne pas exécuter tel quel)
print("release('src/mon_package/__init__.py', 'CHANGELOG.md')")
```

</details>

### Exercice 4 — Système de plugins *(difficile)*

Concevoir un système de plugins basé sur les entry points. Créer :
1. Un package `core` qui découvre et charge des plugins via `importlib.metadata.entry_points()` ;
2. L'interface d'un plugin (protocole ou ABC) ;
3. Un exemple de plugin dans un fichier séparé.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Wheels_et_publication", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
from typing import Protocol
from importlib.metadata import entry_points

# 1. Interface du plugin
class FormatterPlugin(Protocol):
    name: str
    def format(self, data: dict) -> str: ...

# 2. Découverte des plugins
def charger_plugins(group: str = "monapp.formatters") -> dict[str, FormatterPlugin]:
    plugins = {}
    for ep in entry_points(group=group):
        try:
            cls = ep.load()
            plugins[ep.name] = cls()
        except Exception as e:
            print(f"Erreur chargement plugin {ep.name}: {e}")
    return plugins

# 3. Exemple de plugin (serait dans un package séparé)
class JsonFormatter:
    name = "json"
    def format(self, data: dict) -> str:
        import json
        return json.dumps(data, indent=2)

class CsvFormatter:
    name = "csv"
    def format(self, data: dict) -> str:
        return ",".join(f"{k}={v}" for k, v in data.items())

# 4. Utilisation (simulée sans entry points installés)
plugins = {"json": JsonFormatter(), "csv": CsvFormatter()}
data = {"nom": "Alice", "age": 30}
for name, plugin in plugins.items():
    print(f"=== {name} ===")
    print(plugin.format(data))

# En production, les plugins sont découverts via entry_points()
# et installés comme des packages séparés
```

</details>

---

## 12. Ressources

- [Python Packaging User Guide](https://packaging.python.org/)
- [PEP 517 — Build system interface](https://peps.python.org/pep-0517/)
- [PEP 621 — Metadata in pyproject.toml](https://peps.python.org/pep-0621/)
- [Hatch — documentation](https://hatch.pypa.io/)
- [Twine — documentation](https://twine.readthedocs.io/)
- [SemVer 2.0.0](https://semver.org/)